# Initial Downloading, Wrangling, Loading, and Saving Data

To perform the RFM calculation, the data will first be downloaded and cleaned before being loaded for analysis. After the RFM metrics are computed, the resulting dataset will be saved.

Two distinct cleaned datasets will be produced:

1. RFM dataset — contains customer-level RFM metrics.

2. Full cleaned dataset — contains all processed transactional records.

Exploratory Data Analysis (EDA) will be performed on both datasets.

### Initial Imports

In [13]:
import sys
import pandas as pd
import os
import getpass
import kagglehub
import shutil
import json
import zipfile

from pathlib import Path
from sqlalchemy.engine import URL
from sqlalchemy import create_engine, text

sys.path.append(str(Path().resolve().parents[0]))
# Local src
from src.create_database import create_database
from src.clean_column_names import clean_column_names
from src.data_validator import DataValidator


print('Done')

Done


### Step 1: Data Acquisition

In [14]:
cache_path = kagglehub.dataset_download("mashlyn/online-retail-ii-uci", force_download=True)

project_dir = "../data/01_raw"
os.makedirs(project_dir, exist_ok=True)

if cache_path.endswith(".zip"):
    with zipfile.ZipFile(cache_path, 'r') as z:
        z.extractall(project_dir)
    print("Extracted to:", project_dir)
else:
    for filename in os.listdir(cache_path):
        shutil.copy(os.path.join(cache_path, filename), project_dir)
    print("Copied to:", project_dir)

100%|██████████| 14.5M/14.5M [00:03<00:00, 4.29MB/s]

Extracting files...


Copied to: ../data/01_raw


### Loading CSV into DF

In [15]:
path_csv = '../data/01_raw/online_retail_II.csv'
df = pd.read_csv(path_csv)
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


## Step 2: Data Cleaning

In [38]:
df = clean_column_names(df)
df.columns

Index(['invoice', 'stock_code', 'description', 'quantity', 'invoice_date',
       'price', 'customer_id', 'country', 'revenue'],
      dtype='object')

In [17]:
def df_info(df):
    print(f"{'='*30} Head {'='*30}")
    print(df.head())
    print(f"{'='*30} Tail {'='*30}")
    print(df.tail())
    print(f"{'='*30} Shape {'='*30}")
    print(df.shape)
    print(f"{'='*30} Info {'='*30}")
    print(df.info())
    print(f"{'='*30} Columns {'='*30}")
    print(df.columns)
    print(f"{'='*30} Describe {'='*30}")
    print(df.describe())
    print(f"{'='*30} NaN {'='*30}")
    print(df.isnull().sum())
    print(f"{'='*30} Duplicates {'='*30}")
    print(df.duplicated().sum())
df_info(df)

============================== Head ==============================
  invoice stock_code                          description  quantity  \
0  489434      85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434     79323P                   PINK CHERRY LIGHTS        12   
2  489434     79323W                  WHITE CHERRY LIGHTS        12   
3  489434      22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434      21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          invoice_date  price  customer_id         country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  
============================== Tail ==============================
        invoice stock_code                      description  quantity  \


- It shows that price and quantity takes minus values in the dataset; 
- the column customer_id is float;
- the column invoice_date is object;
- 243k missing customer IDs and 4k of descriptions;
- negative values in price & quantity columns;
- 1m+ total records;
- quantities take large numbers 81k;
- 34k of duplicates;

In [18]:
print(df['customer_id'].isnull().sum() / df.shape[0])
print(df['description'].isnull().sum() / df.shape[0])

0.22766872999172733
0.0041054141437232225


- 22.8% of customer IDs are NaNs;
- 0.4% of descriptions are NaNs;

In [19]:
#report = DataValidator.validate(df)

#print(json.dumps(report, indent=2))

Analysis of Negative and Zero Values in Quantities & Prices

Check the frequency of negative or zero values in the dataset to determine their impact and decide on an appropriate handling strategy (e.g., removal or imputation).

In [20]:
print(df.loc[df['quantity'] < 0].shape)
print(df.loc[df['quantity'] == 0].shape)
print(df.loc[df['price'] < 0].shape)
print(df.loc[df['price'] == 0].shape)

(22950, 8)
(0, 8)
(5, 8)
(6202, 8)


- 23k of entries where quantity is negative; either logging/other issues or refunds
- 6k of entries where price is negative; probably logging/other issues

In [21]:
df.loc[df['quantity'] < 0, :]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
1065910,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
1067002,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
1067176,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
1067177,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


- invoice starts with C in quantity < 0, can be mark of cancellation and return;

In [22]:
df.loc[(df['invoice'].str.startswith('C', na=False)) & (df['quantity'] > 0)]

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom


- only one entry with C that has positive quantity;

In [23]:
df[df.duplicated(keep=False)].sort_values(by=list(df.columns))

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
...,...,...,...,...,...,...,...,...
965610,C574510,22360,GLASS JAR ENGLISH CONFECTIONERY,-1,2011-11-04 13:25:00,2.95,15110.0,United Kingdom
986868,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,2011-11-13 11:38:00,0.55,17838.0,United Kingdom
986869,C575940,23309,SET OF 60 I LOVE LONDON CAKE CASES,-24,2011-11-13 11:38:00,0.55,17838.0,United Kingdom
1055441,C580764,22667,RECIPE BOX RETROSPOT,-12,2011-12-06 10:38:00,2.95,14562.0,United Kingdom


- duplicates can be due to logging issues.

In [24]:
df = df.dropna(subset=['customer_id']).copy()
df['customer_id'] = df['customer_id'].astype(int)
#df = df[df['quantity'] > 0].copy()
df = df[df['price'] != 0].copy()
df = df[df['price'] > 0].copy()
df = df.drop(df.loc[(df['invoice'].str.startswith('C', na=False)) & (df['quantity'] > 0)].index).copy()
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['description'] = df['description'].str.strip()
df = df.drop_duplicates().copy()

- dropped empty customer_id columns, as they cannot be filled;
- converted customer_id column type to int for sql loading;
- converted invoice_date column type to datetime for plotting.
- dropped negative and zero values from price column;
- dropped duplicates, as they are probably system log issues;

In [25]:
tmp = df.sort_values(by='price', ascending=False)
print(tmp['description'].head(1000).unique().tolist())
print(tmp['stock_code'].head(1000).unique().tolist())

['Manual', 'POSTAGE', 'Discount', 'DOTCOM POSTAGE', 'CRUK Commission', 'PICNIC BASKET WICKER 60 PIECES', 'Adjustment by john on 26/01/2010 17', 'Adjustment by Peter on 24/05/2010 1', 'Adjustment by Peter on Jun 25 2010', 'Adjustment by john on 26/01/2010 16', 'VINTAGE BLUE KITCHEN CABINET', 'VINTAGE RED KITCHEN CABINET', 'LOVE SEAT ANTIQUE WHITE METAL', 'RUSTIC  SEVENTEEN DRAWER SIDEBOARD', 'REGENCY MIRROR WITH SHUTTERS', 'GIANT SEVENTEEN DRAWER SIDEBOARD', 'CARRIAGE', 'BLUE KASHMIRI COFFEE TABLE', 'CHEST NATURAL WOOD 20 DRAWERS', 'FRENCH STYLE WALL DRESSER', 'ANT WHITE SWEETHEART TABLE W 3 DRAW', 'VINTAGE POST OFFICE CABINET', 'BLUE KASHMIRI OCCASIONAL TABLE', 'PINK KASHMIRI OCCASIONAL TABLE', 'GREEN KASHMIRI OCCASIONAL TABLE', 'SCHOOL DESK AND CHAIR', 'PINK KASHMIRI COFFEE TABLE', 'PINK PAINTED KASHMIRI TABLE', 'BLUE PAINTED KASHMIRI TABLE', 'DECORATIVE HANGING SHELVING UNIT', 'SET/3 COLOUR PAINTED KASHMIRI STOOL', 'BROCADE RING PURSE']
['M', 'POST', 'D', 'DOT', 'CRUK', '22502', 'ADJ

In [26]:
df[df['stock_code'] == 'C2']

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
9292,490127,C2,CARRIAGE,1,2009-12-03 18:13:00,50.0,14156,EIRE
14502,490542,C2,CARRIAGE,1,2009-12-07 09:42:00,50.0,14911,EIRE
19541,490998,C2,CARRIAGE,1,2009-12-08 17:24:00,50.0,16253,United Kingdom
22803,491160,C2,CARRIAGE,1,2009-12-10 10:29:00,50.0,14911,EIRE
32964,492092,C2,CARRIAGE,1,2009-12-15 14:03:00,50.0,14156,EIRE
...,...,...,...,...,...,...,...,...
1038663,579539,C2,CARRIAGE,1,2011-11-30 10:06:00,50.0,14911,EIRE
1040462,579768,C2,CARRIAGE,1,2011-11-30 15:08:00,50.0,14911,EIRE
1041945,579910,C2,CARRIAGE,1,2011-12-01 08:52:00,50.0,14911,EIRE
1044366,580127,C2,CARRIAGE,1,2011-12-01 17:51:00,50.0,14911,EIRE


- many rows with large prices are invalid, the amounts are large (outliers) and as they are not products, it is not suitable for calculations; 

In [27]:
non_product_keywords = [
    'manual', 
    'postage', 
    'dotcom postage', 
    'bank charges', 
    'amazon fee', 
    'discount', 
    'cruk commission', 
    'adjust',
    'carriage'
]

pattern = '|'.join(non_product_keywords)

df = df[~df['description'].str.lower().str.contains(pattern, na=False)].copy()

- dropped non-product transactions

In [28]:
tmp = df.sort_values(by='quantity', ascending=False)
print(tmp[['description', 'stock_code', 'price', 'quantity']].head(50))
print(tmp[['description', 'stock_code', 'price', 'quantity']].tail(50))

                                 description stock_code  price  quantity
1065882          PAPER CRAFT , LITTLE BIRDIE      23843   2.08     80995
587080        MEDIUM CERAMIC TOP STORAGE JAR      23166   1.04     74215
90857     BLACK AND WHITE PAISLEY FLOWER MUG      37410   0.10     19152
127168           SET/6 WOODLAND PAPER PLATES      21091   0.10     12960
127166           SET/6 STRAWBERRY PAPER CUPS      21099   0.10     12960
127169             SET/6 WOODLAND PAPER CUPS      21085   0.10     12744
127167         SET/6 STRAWBERRY PAPER PLATES      21092   0.10     12480
135027       PACK OF 12 PINK PAISLEY TISSUES      21984   0.25     10000
135029         PACK OF 12 RED SPOTTY TISSUES      21980   0.25     10000
135028               PACK OF 12 SUKI TISSUES      21982   0.25     10000
135030           PACK OF 12 WOODLAND TISSUES      21981   0.25     10000
93677        SMALL FAIRY CAKE FRIDGE MAGNETS      85220   0.30      9456
432176   ROTATING SILVER ANGELS T-LIGHT HLDR      8

- big positive & negative quantities are valid, as they capture regular purchases & returns;

In [29]:
df['revenue'] = df['quantity'] * df['price']
df.groupby('customer_id')['revenue'].sum().describe()

count      5875.000000
mean       2784.032398
std       13798.791404
min       -1343.240000
25%         328.000000
50%         835.220000
75%        2164.280000
max      578408.640000
Name: revenue, dtype: float64

- some of our customers are having negative revenue, probably data on their initial purchase was not collected, but return was;

In [30]:
tmp = df.groupby('customer_id')['revenue'].sum().reset_index()
negative_ids = tmp['customer_id'][tmp['revenue'] < 0]
df = df[~df['customer_id'].isin(negative_ids)].copy()
df

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00
...,...,...,...,...,...,...,...,...,...
1067365,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,10.20
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60


- dropped all transactions of customers that would have negative revenue balance in the monetary from the main df;

In [31]:
print(df[['country', 'revenue', 'invoice_date']].sort_values(by='invoice_date', ascending=False).head(50))
print(df[['country', 'revenue', 'invoice_date']].sort_values(by='invoice_date', ascending=False).tail(50))

                country  revenue        invoice_date
1067369          France    14.85 2011-12-09 12:50:00
1067362          France    15.60 2011-12-09 12:50:00
1067356          France    19.80 2011-12-09 12:50:00
1067357          France    19.80 2011-12-09 12:50:00
1067358          France    15.00 2011-12-09 12:50:00
1067359          France    15.00 2011-12-09 12:50:00
1067360          France    15.00 2011-12-09 12:50:00
1067361          France    15.00 2011-12-09 12:50:00
1067355          France    23.40 2011-12-09 12:50:00
1067363          France    23.40 2011-12-09 12:50:00
1067365          France    10.20 2011-12-09 12:50:00
1067366          France    12.60 2011-12-09 12:50:00
1067367          France    16.60 2011-12-09 12:50:00
1067368          France    16.60 2011-12-09 12:50:00
1067364          France    16.60 2011-12-09 12:50:00
1067353  United Kingdom   214.80 2011-12-09 12:49:00
1067352  United Kingdom    30.00 2011-12-09 12:49:00
1067351  United Kingdom    23.60 2011-12-09 12

- the records start and end smoothly, in Dec, right before New Year;

In [37]:
df_info(df)

============================== Head ==============================
  invoice stock_code                          description  quantity  \
0  489434      85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434     79323P                   PINK CHERRY LIGHTS        12   
2  489434     79323W                  WHITE CHERRY LIGHTS        12   
3  489434      22041          RECORD FRAME 7" SINGLE SIZE        48   
4  489434      21232       STRAWBERRY CERAMIC TRINKET BOX        24   

         invoice_date  price  customer_id         country  revenue  
0 2009-12-01 07:45:00   6.95        13085  United Kingdom     83.4  
1 2009-12-01 07:45:00   6.75        13085  United Kingdom     81.0  
2 2009-12-01 07:45:00   6.75        13085  United Kingdom     81.0  
3 2009-12-01 07:45:00   2.10        13085  United Kingdom    100.8  
4 2009-12-01 07:45:00   1.25        13085  United Kingdom     30.0  
============================== Tail ==============================
        invoice stock_cod

In [33]:
pd_to_csv_path = os.path.join('..', 'data', '02_processed', 'cleaned_retail.csv')

os.makedirs(os.path.dirname(pd_to_csv_path), exist_ok=True)
df.to_csv(pd_to_csv_path, index=False)

print(f"Saved to: {pd_to_csv_path}")

Saved to: ../data/02_processed/cleaned_retail.csv


- saved for further eda;

## Step 3: Database Ingestion for Feature Engineering (RFM)

In [34]:
db_name = input(str('Enter your database name: '))
password = getpass.getpass("Enter your database password: ")

base_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432
)

create_database(base_url, db_name)

engine = create_engine(base_url.set(database=db_name))


df.to_sql(                  # Can be used sql to load, if dataset is much bigger
    'retail_records', 
    engine, 
    if_exists='replace'
)

Database 'retail' created successfully.


755

- created database & table, and loaded dataset in for calculations;

In [35]:
rfm_query = """
WITH cleaned AS (
    SELECT 
        customer_id,
        invoice,
        CAST(invoice_date AS TIMESTAMP) as invoice_date,
        quantity,
        price
    FROM public.retail_records
    WHERE quantity > 0
      AND price > 0
      AND customer_id IS NOT NULL
),
rfm_base AS (
    SELECT
        customer_id,
        MAX(invoice_date) AS last_purchase_date,
        COUNT(DISTINCT invoice) AS frequency,
        SUM(quantity * price) AS monetary
    FROM cleaned
    GROUP BY customer_id
)
SELECT
    customer_id,
    last_purchase_date,
    frequency,
    monetary,
    DATE_PART('day', (SELECT MAX(last_purchase_date) FROM rfm_base) - last_purchase_date) AS recency,
    NTILE(5) OVER (ORDER BY DATE_PART('day', (SELECT MAX(last_purchase_date) FROM rfm_base) - last_purchase_date) DESC) AS R_score,
    NTILE(5) OVER (ORDER BY frequency ASC) AS F_score,
    NTILE(5) OVER (ORDER BY monetary ASC) AS M_score
FROM rfm_base;
"""

with engine.connect() as conn:
    df_rfm = pd.read_sql(text(rfm_query), conn)

df_rfm

,customer_id,last_purchase_date,frequency,monetary,recency,r_score,f_score,m_score
0,17592,2009-12-01 10:49:00,1,148.30,738.0,1,1,1
1,13526,2009-12-01 13:13:00,2,1182.00,737.0,1,3,3
2,17056,2009-12-01 12:55:00,1,128.60,737.0,1,1,1
3,15833,2009-12-02 11:59:00,1,80.40,737.0,1,1,1
4,14654,2009-12-01 12:57:00,1,246.86,737.0,1,1,1
...,...,...,...,...,...,...,...,...
5842,17491,2011-12-08 16:18:00,17,5653.67,0.0,5,5,5
5843,17490,2011-12-09 09:08:00,9,2750.50,0.0,5,5,4
5844,12526,2011-12-09 12:09:00,3,1172.66,0.0,5,3,3
5845,17428,2011-12-09 09:45:00,45,31466.76,0.0,5,5,5


- performed rfm calculations in sql and unloaded it into df for saving as csv;

In [36]:
pd_to_csv_path = os.path.join('..', 'data', '02_processed', 'rfm_retail.csv')

os.makedirs(os.path.dirname(pd_to_csv_path), exist_ok=True)
df_rfm.to_csv(pd_to_csv_path, index=False)

print(f"Saved to: {pd_to_csv_path}")

Saved to: ../data/02_processed/rfm_retail.csv


- saved for further eda and segmentation;